# Session 10 - Sampling Techniques


In [1]:
import numpy as np
import random
from collections import Counter, defaultdict

random.seed(1)
np.random.seed(2)

## Task 1 — Random Sample of 20 Users from 200 IDs (Spotify Survey)

In [2]:
user_ids = list(range(1, 201))
survey_sample = random.sample(user_ids, 20)

print('Selected user IDs:', sorted(survey_sample))
print('Sample size:', len(survey_sample))

Selected user IDs: [8, 17, 25, 31, 35, 54, 66, 98, 100, 111, 116, 121, 125, 127, 146, 156, 167, 195, 196, 197]
Sample size: 20


## Task 2 — Stratified Sample of 60 Flipkart Orders by City

1000 orders are generated with city labels, then a sample of 60 is drawn so each city's share of the sample matches its share of the full dataset. Proportions are converted to whole numbers using the largest-remainder method so the sample always totals exactly 60.

In [3]:
cities = ['Ahmedabad', 'Surat', 'Vadodara']
city_weights = [0.40, 0.35, 0.25]
orders_city = np.random.choice(cities, size=1000, p=city_weights)

city_counts = Counter(orders_city)
print('City counts in full dataset:', dict(city_counts))

total_sample = 60
raw_alloc = {c: (n / 1000) * total_sample for c, n in city_counts.items()}
allocation = {c: int(v) for c, v in raw_alloc.items()}
remainder = total_sample - sum(allocation.values())
ranked = sorted(raw_alloc, key=lambda c: raw_alloc[c] - allocation[c], reverse=True)
for i in range(remainder):
    allocation[ranked[i]] += 1

print('Proportional allocation:', allocation, '| total:', sum(allocation.values()))

city_indices = defaultdict(list)
for idx, city in enumerate(orders_city):
    city_indices[city].append(idx)

stratified_sample_idx = []
for city, n in allocation.items():
    stratified_sample_idx.extend(random.sample(city_indices[city], n))

print('Final stratified sample size:', len(stratified_sample_idx))

City counts in full dataset: {np.str_('Surat'): 360, np.str_('Ahmedabad'): 416, np.str_('Vadodara'): 224}
Proportional allocation: {np.str_('Surat'): 22, np.str_('Ahmedabad'): 25, np.str_('Vadodara'): 13} | total: 60
Final stratified sample size: 60


## Task 3 — Systematic Sampling of 500 Zomato Reviews (Every 10th)

In [4]:
def systematic_sample(data, k):
    return data[::k]

reviews = [f'Review_{i}' for i in range(1, 501)]
review_sample = systematic_sample(reviews, 10)

print('Sample size:', len(review_sample))
print('First 5:', review_sample[:5])
print('Last 5:', review_sample[-5:])

Sample size: 50
First 5: ['Review_1', 'Review_11', 'Review_21', 'Review_31', 'Review_41']
Last 5: ['Review_451', 'Review_461', 'Review_471', 'Review_481', 'Review_491']


## Task 4 — How Sample Size Affects Estimate Accuracy (Swiggy Delivery Times)

Accuracy of a sample mean is measured by its **standard error**: $SE = \dfrac{\sigma}{\sqrt{n}}$. Assuming an illustrative population standard deviation of σ = 8 minutes across the 10,000 delivery records (not given, so assumed for this calculation), here's how the standard error and the 95% confidence margin shrink as sample size grows:

In [5]:
population_std = 8
sample_sizes = [50, 200, 500]

for n in sample_sizes:
    se = population_std / (n ** 0.5)
    margin_95 = 1.96 * se
    print(f'n={n:4d}  SE={se:.4f} min  95% margin=±{margin_95:.4f} min')

n=  50  SE=1.1314 min  95% margin=±2.2175 min
n= 200  SE=0.5657 min  95% margin=±1.1087 min
n= 500  SE=0.3578 min  95% margin=±0.7012 min


**Interpretation:** Going from 50 to 200 records (4x the sample) roughly **halves** the margin of error — from about ±2.22 min to ±1.11 min — because standard error scales with $1/\sqrt{n}$, not $1/n$. Going from 200 to 500 (2.5x) only tightens it further to about ±0.70 min. This is the diminishing-returns pattern behind sample size: doubling precision requires roughly quadrupling the sample, so past a certain point a much bigger sample buys only a small accuracy gain — useful to know before assuming 'more data' is always worth the extra cost.

## Task 5 — Stratified Sample of 40 IPL Fans by Favorite Team (Own Logic)

This intentionally uses a different approach from the Task 2 demo-style code above: fans are grouped with a plain loop instead of `Counter`, and the sample-size allocation is computed by cycling through fractional remainders rather than a one-pass sort.

In [6]:
teams = ['CSK', 'MI', 'RCB', 'KKR', 'GT', 'RR']
team_weights = [0.22, 0.20, 0.20, 0.15, 0.13, 0.10]

fans = []
for _ in range(800):
    r = random.random()
    cum = 0
    for team, w in zip(teams, team_weights):
        cum += w
        if r <= cum:
            fans.append(team)
            break

groups = {}
for team in fans:
    groups.setdefault(team, []).append(team)

total_fans = len(fans)
target_sample = 40

exact_shares = {team: (len(members) / total_fans) * target_sample for team, members in groups.items()}
allocation = {team: int(share) for team, share in exact_shares.items()}
leftover = target_sample - sum(allocation.values())

fractions = {team: exact_shares[team] - allocation[team] for team in groups}
ranked_by_fraction = sorted(fractions, key=fractions.get, reverse=True)
for i in range(leftover):
    allocation[ranked_by_fraction[i % len(ranked_by_fraction)]] += 1

print('Fan counts per team:', {t: len(m) for t, m in groups.items()})
print('Proportional sample allocation:', allocation, '| total:', sum(allocation.values()))

final_sample = []
for team, count in allocation.items():
    team_members = groups[team]
    picks = random.sample(range(len(team_members)), count)
    final_sample.extend([(team, p) for p in picks])

print('Final survey sample size:', len(final_sample))

Fan counts per team: {'RCB': 168, 'KKR': 104, 'MI': 165, 'GT': 102, 'CSK': 178, 'RR': 83}
Proportional sample allocation: {'RCB': 9, 'KKR': 5, 'MI': 8, 'GT': 5, 'CSK': 9, 'RR': 4} | total: 40
Final survey sample size: 40
